# Evaluate Supply Chain Supervisor Agent
This notebook evaluates the **Multi-Agent Supervisor (MAS)** that routes between:
- **`genie_data_agent`** — the Genie space over `nodes`/`edges`/`bom` (data-lookup questions), and
- **`optimization_agent`** — the MCP optimization server (disruption / TTR / TTS / recommendation questions).

It mirrors `02_evaluate_agent.ipynb`, but the tool-usage scorer becomes a **routing** scorer: instead of checking `data_analysis_tool` vs `optimization_tool`, it checks which sub-agent the supervisor routed to.

## Cluster Configuration
Tested on Databricks Runtime 17.3 LTS ML.

In [ ]:
%pip install -q "mlflow[databricks]==3.12.0" "databricks-agents==1.10.2" "databricks-sdk==0.94.0"
dbutils.library.restartPython()

## Configuration
Set the deployed Supervisor Agent serving endpoint name (from `manage_mas` / the Agent Bricks UI).

In [ ]:
# Deployed Supervisor Agent endpoint (created via manage_mas on 2026-07-02)
MAS_ENDPOINT_NAME = 'mas-3dd7e8a7-endpoint'  # 'Supply Chain Supervisor' MAS endpoint

# Sub-agent names as configured in the supervisor
GENIE_AGENT = 'genie_data_agent'
OPTIMIZATION_AGENT = 'optimization_agent'

In [ ]:
import mlflow
mlflow.set_registry_uri('databricks-uc')
user_email = spark.sql('select current_user() as user').collect()[0]['user']
mlflow.set_experiment(f'/Users/{user_email}/supply-chain-supervisor')

## Query helper
Call the Supervisor Agent serving endpoint. The MAS accepts an OpenAI-style `messages` payload and returns the routed agent's answer.

In [ ]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client('databricks')

def query_supervisor(messages, max_rounds=4):
    """Call the Supervisor Agent (ResponsesAgent schema: `input`).

    The MAS gates external MCP tool calls behind `mcp_approval_request` items
    (human-in-the-loop). We auto-approve them and continue the conversation so
    the routed sub-agent actually executes and the final answer is returned.
    """
    conversation = list(messages)
    resp = client.predict(endpoint=MAS_ENDPOINT_NAME, inputs={'input': conversation})
    for _ in range(max_rounds):
        output = resp.get('output', [])
        approvals = [i for i in output if i.get('type') == 'mcp_approval_request']
        if not approvals:
            break
        conversation = conversation + output + [
            {'type': 'mcp_approval_response',
             'approval_request_id': a['id'],
             'approve': True}
            for a in approvals
        ]
        resp = client.predict(endpoint=MAS_ENDPOINT_NAME, inputs={'input': conversation})
    return resp

## Test the supervisor
Send a disruption question and a data-lookup question to sanity-check routing.

In [ ]:
query_supervisor([{'role':'user','content':'What happens if T2_4 goes down and takes 6 weeks to recover? What should I do?'}])

In [ ]:
query_supervisor([{'role':'user','content':'Tell me the demand for T1_5 and the inventory levels of all materials needed to produce this finished good.'}])

## Evaluation dataset
The same 5 questions as `02_evaluate_agent.ipynb`, but `expected_tool` is replaced by `expected_agent`: data-lookup questions should route to the Genie agent; disruption/recovery questions to the optimization agent.

In [ ]:
eval_data = [
    {'inputs': {'messages': [{'role':'user','content':'List all downstream production sites for the raw material supplied by T3_10, and include any related information about these sites.'}]},
     'expectations': {'expected_agent': [GENIE_AGENT]}},
    {'inputs': {'messages': [{'role':'user','content':'Tell me what happens if T2_8 is disrupted and requires 9 to recover. What should I do?'}]},
     'expectations': {'expected_agent': [OPTIMIZATION_AGENT]}},
    {'inputs': {'messages': [{'role':'user','content':'What happens if T2_4 goes down and takes 6 weeks to recover? What are the recommendations?'}]},
     'expectations': {'expected_agent': [OPTIMIZATION_AGENT]}},
    {'inputs': {'messages': [{'role':'user','content':'Tell me the demand for T1_5 and the inventory levels of all materials needed to produce this finished goods.'}]},
     'expectations': {'expected_agent': [GENIE_AGENT]}},
    {'inputs': {'messages': [{'role':'user','content':'There has been an incident at T3_15 and it will go down for the next 10 time units. What can I do to mitigate the risk?'}]},
     'expectations': {'expected_agent': [OPTIMIZATION_AGENT]}},
]

## Generate traces
Run the supervisor over the dataset; traces are logged to the experiment above.

In [ ]:
from mlflow.genai.scorers import scorer

def evaluate_model(messages) -> dict:
    return query_supervisor(messages)

@scorer
def dummy_metric():
    return 1

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=evaluate_model,
    scorers=[dummy_metric],
)

In [ ]:
generated_traces = mlflow.search_traces(run_id=results.run_id)

## Routing scorer
Inspects the trace to determine which sub-agent the supervisor invoked, and compares against `expected_agent`.
The supervisor exposes routing either as CHAIN/AGENT/TOOL spans named after the sub-agent, or as a routed-agent field in span attributes; the scorer scans span names and I/O for the expected agent name so it is robust to the exact trace shape.

In [ ]:
import json
from typing import Any
from mlflow.entities import Trace, Feedback, SpanType

def _routed_agents(trace: Trace) -> set[str]:
    """Best-effort extraction of the sub-agent(s) the supervisor routed to."""
    names = set()
    for span in trace.search_spans():
        blob = json.dumps(span.to_dict(), ensure_ascii=False).lower()
        for agent in (GENIE_AGENT, OPTIMIZATION_AGENT):
            if agent.lower() in blob:
                names.add(agent)
        # Genie routing often shows as a genie/sql span; optimization as an mcp/tool call
        sn = (span.name or '').lower()
        if 'genie' in sn or 'sql' in sn:
            names.add(GENIE_AGENT)
        if 'stress_test' in blob or 'run_supply_chain' in blob or 'optimiz' in sn:
            names.add(OPTIMIZATION_AGENT)
    return names

@scorer
def routing(expectations: dict[str, Any], trace: Trace) -> Feedback:
    expected = set(expectations.get('expected_agent', []))
    routed = _routed_agents(trace)
    missing = expected - routed
    if not missing:
        return Feedback(value='yes', rationale=f'Routed to expected agent(s): {sorted(expected)} (observed {sorted(routed)}).')
    return Feedback(value='no', rationale=f'Expected {sorted(expected)} but observed routing {sorted(routed)}.')

## Quality scorers
Carried over unchanged from `02_evaluate_agent.ipynb`.

In [ ]:
from mlflow.genai.scorers import Guidelines, RelevanceToQuery
from mlflow.entities import SpanType

@scorer
def response_time(trace: Trace) -> Feedback:
    agent_spans = trace.search_spans(span_type=SpanType.AGENT)
    root = agent_spans[0] if agent_spans else trace.search_spans()[0]
    secs = (root.end_time_ns - root.start_time_ns) / 1e9
    max_duration = 120
    if secs <= max_duration:
        return Feedback(value='yes', rationale=f'Response time {secs:.2f}s within {max_duration}s.')
    return Feedback(value='no', rationale=f'Response time {secs:.2f}s exceeds {max_duration}s.')

scorers = [
    Guidelines(name='response_length', guidelines='The response MUST be concise and to the point not longer than 500 words.'),
    Guidelines(name='professional_tone', guidelines='The response MUST be in a professional tone.'),
    Guidelines(name='includes_recommendations', guidelines='For disruption/recovery questions, the response MUST include specific, actionable recommendations.'),
    RelevanceToQuery(),
    response_time,
    routing,
]

## Run evaluation with the custom scorers

In [ ]:
trace_check_eval_results = mlflow.genai.evaluate(
    data=generated_traces,
    scorers=scorers,
)

## Next steps
Review the routing and quality metrics in the MLflow experiment. If routing is wrong for any question, refine the supervisor's `instructions` / agent `description`s (via `manage_mas`) and re-run. Once satisfied, the supervisor endpoint is ready to embed in the frontend application.